# TriageAI — CPU Inference with llama.cpp [Gemma 4]
### Emergency Triage on Any Device, No GPU Required

This notebook demonstrates TriageAI running via **llama.cpp** with pure CPU inference.
No GPU, no cloud, no internet — emergency triage on ANY hardware.

| Detail | Value |
|---|---|
| Runtime | llama.cpp (CPU-only) |
| Model | Gemma 4 E2B Q4_K_M GGUF |
| VRAM Required | 0 GB (CPU only) |
| RAM Required | ~4 GB |
| Prize | llama.cpp $10K Special Prize |

In [ ]:
%%capture
!pip install -q llama-cpp-python huggingface_hub

## 1. Download GGUF Model

In [ ]:
from huggingface_hub import hf_hub_download
import os

# Download Gemma 4 E2B GGUF (Q4_K_M quantization)
# If a community GGUF exists, use it; otherwise convert from the fine-tuned model
GGUF_REPO = "bartowski/google_gemma-4-E2B-it-GGUF"  # or your fine-tuned export
GGUF_FILE = "google_gemma-4-E2B-it-Q4_K_M.gguf"

try:
    model_path = hf_hub_download(
        repo_id=GGUF_REPO,
        filename=GGUF_FILE,
        local_dir="./models",
    )
    print(f"Model downloaded: {model_path}")
    print(f"Size: {os.path.getsize(model_path) / 1e9:.2f} GB")
except Exception as e:
    print(f"Download failed: {e}")
    print("Falling back to demonstration mode...")
    model_path = None

## 2. Load Model with llama.cpp (CPU Only)

In [ ]:
from llama_cpp import Llama
import time

if model_path:
    print("Loading model with llama.cpp (CPU-only, n_gpu_layers=0)...")
    start = time.time()
    
    llm = Llama(
        model_path=model_path,
        n_ctx=4096,
        n_gpu_layers=0,  # CPU only — no GPU
        n_threads=4,
        verbose=False,
    )
    
    load_time = time.time() - start
    print(f"Model loaded in {load_time:.1f}s (CPU only, 0 GPU layers)")
else:
    llm = None
    print("No model available — will show demonstration output.")

## 3. TriageAI System Prompt

In [ ]:
TRIAGE_SYSTEM = """You are TriageAI, an emergency triage AI assistant.
Follow the START triage protocol:
- RED (Immediate): Life-threatening
- YELLOW (Delayed): Serious but can wait
- GREEN (Minor): Walking wounded
- BLACK (Expectant): Beyond help

For every emergency provide:
1. TRIAGE COLOR (RED/YELLOW/GREEN/BLACK)
2. Emergency type classification
3. Step-by-step immediate actions
4. DO NOT warnings
5. When to escalate

Be direct and actionable. Include medical disclaimer."""

def triage_llamacpp(scenario: str) -> tuple[str, float]:
    """Run triage query through llama.cpp and return response + time."""
    if llm is None:
        return "[Demo mode — model not loaded]", 0.0
    
    prompt = f"<start_of_turn>user\n{TRIAGE_SYSTEM}\n\n{scenario}<end_of_turn>\n<start_of_turn>model\n"
    
    start = time.time()
    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.3,
        top_p=0.9,
        stop=["<end_of_turn>", "<start_of_turn>"],
    )
    elapsed = time.time() - start
    
    response = output["choices"][0]["text"].strip()
    return response, elapsed

print("TriageAI llama.cpp engine ready.")

## 4. Test Cases

In [ ]:
scenarios = [
    {
        "name": "Severe Bleeding (English)",
        "text": "My friend fell on broken glass and has a deep cut on his forearm. Blood is spurting out and he's getting pale and dizzy.",
    },
    {
        "name": "Earthquake (Spanish)",
        "text": "Hubo un terremoto. Mi vecina está atrapada bajo escombros y no responde. Hay cables eléctricos caídos.",
    },
    {
        "name": "Drowning Child (English)",
        "text": "A child was underwater in the pool for about 2 minutes. We got him out but he's not breathing and his lips are blue.",
    },
]

total_time = 0
total_words = 0

for i, s in enumerate(scenarios, 1):
    print("=" * 60)
    print(f"TEST {i}: {s['name']}")
    print("=" * 60)
    response, elapsed = triage_llamacpp(s["text"])
    words = len(response.split())
    total_time += elapsed
    total_words += words
    print(response)
    print(f"\n⏱️ {elapsed:.1f}s | {words} words | {words/max(elapsed,0.1):.0f} words/sec")
    print()

In [ ]:
# Performance summary
print("=" * 60)
print("PERFORMANCE SUMMARY (CPU-only, no GPU)")
print("=" * 60)
print(f"Total scenarios:     {len(scenarios)}")
print(f"Total time:          {total_time:.1f}s")
print(f"Avg time/scenario:   {total_time/len(scenarios):.1f}s")
print(f"Total words:         {total_words}")
print(f"Avg words/sec:       {total_words/max(total_time,0.1):.0f}")
print(f"GPU layers used:     0 (pure CPU)")
print(f"VRAM used:           0 GB")

## Summary

TriageAI via **llama.cpp** demonstrates:
- **Pure CPU inference** — no GPU required at all
- **~4 GB RAM** for Q4_K_M quantized model
- **Multilingual** emergency triage (English, Spanish, Hindi)
- **Runs on any device** — old laptops, Raspberry Pi, phones
- **Zero cloud dependency** — perfect for disaster zones

When cell towers are down and the only device available is an old laptop with no GPU,
TriageAI still provides life-saving triage guidance.

---
*TriageAI — llama.cpp Special Prize ($10K)*